In [5]:
%idle_timeout 30
%glue_version 4.0
%worker_type G.1X
%number_of_workers 2

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Current idle_timeout is None minutes.
idle_timeout has been set to 30 minutes.
Setting Glue version to: 4.0
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 2


In [1]:
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.dynamicframe import DynamicFrame
from awsglue.job import Job
from pyspark.sql import DataFrame
from pyspark.sql.functions import (
    array, struct, explode, col, to_timestamp, year, lit,
    regexp_extract, split, length, regexp_replace, trim, monotonically_increasing_id
)
from functools import reduce

print('Bibliotecas importadas!')

Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 2
Idle Timeout: 30
Session ID: 7600755d-9841-4c0f-a007-4545efcf99d7
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
Waiting for session 7600755d-9841-4c0f-a007-4545efcf99d7 to get into ready status...
Session 7600755d-9841-4c0f-a007-4545efcf99d7 has been created.
Bibliotecas importadas!


In [12]:
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

job = Job(glueContext)
job.init("teste-interativo-silver")

DATABASE = "state_of_data_db"
BUCKET = "pos-tech-state-of-data-grupo-72"
anos = [2021, 2022, 2023, 2024, 2025]
print(anos)

[2021, 2022, 2023, 2024, 2025]


In [8]:
dfs = list()

for ano in anos:
    df = spark.read \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .option("multiLine", "true") \
        .option("escape", "\"") \
        .csv(f"s3://{BUCKET}/bronze/state-of-data/{ano}/")
    
    if '0.d_data/hora_envio' in df.columns:
        df = df.withColumn(
            'ano',
            year(to_timestamp(col('`0.d_data/hora_envio`'), 'dd/MM/yyyy HH:mm:ss'))
        )
    else:
        df = df.withColumn('ano', lit(ano))
    print(f"Lido: ano de {ano} com sucesso!")
    dfs.append(df)

Lido: ano de 2021 com sucesso!
Lido: ano de 2022 com sucesso!
Lido: ano de 2023 com sucesso!
Lido: ano de 2024 com sucesso!
Lido: ano de 2025 com sucesso!


In [14]:
# Pivotando as colunas em linhas para facilitar os tratamentos de dados e transformar os códigos das perguntas e separando os dfs em cada tipo de tratamento

# Adiciona o ID técnico em cada df, ANTES do melt — não depende do nome/formato do id original de cada ano
for i in range(len(dfs)):
    dfs[i] = dfs[i].withColumn("id_resposta", monotonically_increasing_id())

def melt_spark(df: DataFrame, id_vars, var_name="pergunta", value_name="resposta") -> DataFrame:
    value_vars = [c for c in df.columns if c not in id_vars]
    vars_and_vals = array(*[
        struct(
            lit(c).alias(var_name),
            col(f"`{c}`").cast("string").alias(value_name)
        )
        for c in value_vars
    ])
    tmp = df.select(*id_vars, explode(vars_and_vals).alias("vars_and_vals"))
    return tmp.select(
        *id_vars,
        col(f"vars_and_vals.{var_name}").alias(var_name),
        col(f"vars_and_vals.{value_name}").alias(value_name),
    )

# id_vars agora inclui id_resposta, junto com ano
melted_dfs = [melt_spark(df, id_vars=['ano', 'id_resposta']) for df in dfs]
df_final = reduce(lambda a, b: a.unionByName(b), melted_dfs)

# Separando os dfs em cada tipo de tratamento

df_parenteses = df_final.filter(col("pergunta").startswith("('"))
df_numero = df_final.filter(col("pergunta").rlike(r"^\d"))
df_outros = df_final.filter(
    ~col("pergunta").startswith("('") &
    ~col("pergunta").rlike(r"^\d")
)

print(df_final.count())
print(df_parenteses.count())
print(df_numero.count())
print(df_outros.count())

8019701
4561190
3458511
0


In [15]:
# Tratamento do df que começa com "("

df_parenteses = df_parenteses.withColumn(
    "codigo", regexp_extract(col("pergunta"), r"^\('(.*?)', '(.*)'\)$", 1)
).withColumn(
    "pergunta", regexp_extract(col("pergunta"), r"^\('(.*?)', '(.*)'\)$", 2)
)

df_parenteses = df_parenteses.withColumn(
    "codigo_split", split(col("codigo"), "_")
).withColumn(
    "codigo_1", col("codigo_split")[0]
).withColumn(
    "codigo_2", col("codigo_split")[1]
).withColumn(
    "codigo_3", col("codigo_split")[2]
).drop("codigo_split")

print('df_parenteses tratado!')

df_parenteses tratado!


In [16]:
# Tratamento do df que começa com numeros

df_numero = df_numero.withColumn(
    "split_1", split(col("pergunta"), "_", 2)
).withColumn(
    "codigo", col("split_1")[0]
).withColumn(
    "pergunta", col("split_1")[1]
).drop("split_1")

df_numero_ok = df_numero.filter(length(col("codigo")) <= 7)
df_sem_underline = df_numero.filter(length(col("codigo")) > 7)
df_espaco = df_sem_underline.filter(length(col("codigo")) > 7)

df_espaco = df_espaco.withColumn(
    "split_2", split(col("codigo"), " ", 2)
).withColumn(
    "codigo", col("split_2")[0]
).withColumn(
    "pergunta", col("split_2")[1]
).drop("split_2")

df_sem_underline_espaco = df_espaco.filter(length(col("codigo")) > 7)

df_numero_ok = df_numero_ok.unionByName(df_espaco)

df_numero_ok = df_numero_ok.withColumn(
    "split_3", split(col("codigo"), r"\.")
).withColumn(
    "codigo_1", col("split_3")[0]
).withColumn(
    "codigo_2", col("split_3")[1]
).withColumn(
    "codigo_3", col("split_3")[2]
).drop("split_3")

print('df_numero tratado!')

df_numero tratado!


In [20]:
# Unindo os dois dataframes tratados e transformando em DynamicFrame

df_corrigido = df_parenteses.unionByName(df_numero_ok)

print(df_corrigido.count())
print(df_final.count())

df_corrigido = df_corrigido.withColumn(
    "codigo_1", regexp_replace(col("codigo_1"), r"\D", "")
)
df_corrigido = df_corrigido.withColumn(
    "pergunta", regexp_replace(col("pergunta"), '""', '"')
)
df_corrigido = df_corrigido.withColumn(
    "pergunta", regexp_replace(col("pergunta"), '""', '"')
)

df_corrigido = df_corrigido.select(
    [trim(col(c)).alias(c) if dtype == "string" else col(c)
     for c, dtype in df_corrigido.dtypes]
)

dyf_final = DynamicFrame.fromDF(df_corrigido, glueContext, "dyf_final")

print('df_corrigido (versao final) criado e transformado em DynamicFrame em dyf_final')

8019701
8019701
df_corrigido (versao final) criado e transformado em DynamicFrame em dyf_final


In [23]:
# Salvando no S3 e catalogando no Data Catalog

caminho_silver = f"s3://{BUCKET}/silver/state-of-data/"

# Limpa qualquer dado anterior antes de escrever (evita duplicação por múltiplas execuções)
glueContext.purge_s3_path(caminho_silver, options={"retentionPeriod": 0})

sink = glueContext.getSink(
    connection_type="s3",
    path=caminho_silver,
    enableUpdateCatalog=True,
    partitionKeys=["ano"]
)
sink.setFormat("glueparquet")
sink.setCatalogInfo(catalogDatabase=DATABASE, catalogTableName="silver_state_of_data")
sink.writeFrame(dyf_final)

print("Silver gravada e catalogada com sucesso.")
job.commit()

Silver gravada e catalogada com sucesso.
